# URL TCN and RF/TCN ensemble
The tokenizer is fixed UTF-8 bytes, so training and ONNX inference cannot silently use different vocabularies.

In [ ]:
# Run once, restart the kernel, and confirm CUDA below.
%pip install -r ../requirements-tcn.txt

In [ ]:
from pathlib import Path
import sys
import torch
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from persianphish_detector.models.train_tcn import train_tcn
if not torch.cuda.is_available():
    raise RuntimeError('Use the CUDA environment on the 5070 Ti host for this training cell.')
report = train_tcn(
    ROOT / 'data/processed/realworld_v3_features.csv',
    ROOT / 'artifacts/v3/detector_v3.joblib',
    ROOT / 'artifacts/notebook/url_tcn.onnx',
    ROOT / 'artifacts/notebook/detector_v3_tcn.joblib',
    epochs=20, patience=4, batch_size=256,
)
report['tcn_test_at_0_5'], report['ensemble_test_at_policy_threshold']

In [ ]:
from persianphish_detector.models.tcn import ONNXTCNPredictor
predictor = ONNXTCNPredictor(ROOT / 'artifacts/notebook/url_tcn.onnx')
for url in ['https://soft98.ir/', 'https://secure-login-paypa1.example/verify']:
    print(url, predictor.predict(url))